In [1]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [ ]:
chicago_avg_price_gas = pd.read_csv("chicago_avg_price_gas.csv")
chicago_cta_daily_boarding = pd.read_csv("chicago_cta_daily_boarding.csv")
chicago_weather_2425 = pd.read_csv("chicago_weather_2425.csv")

In [ ]:
chicago_weather_2425

In [ ]:
chicago_weather_2425['DATE'] = pd.to_datetime(chicago_weather_2425['DATE'])
weather_small = chicago_weather_2425[['DATE', 'PRCP', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN',
                                      'WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06',
                                      'WT07', 'WT08', 'WT09', 'WT10', 'WT11']]
wt_cols = ['WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06',
           'WT07', 'WT08', 'WT09', 'WT10', 'WT11']

weather_small[wt_cols] = weather_small[wt_cols].fillna(0)

weather_daily = weather_small.groupby('DATE')[['PRCP', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN']].mean()
weather_events = weather_small.groupby('DATE')[['WT01', 'WT02', 'WT03', 'WT04', 'WT05', 'WT06',
                                                'WT07', 'WT08', 'WT09', 'WT10', 'WT11']].max()
weather_daily = pd.concat([weather_daily, weather_events], axis=1).reset_index()
weather_daily['month'] = weather_daily['DATE'].dt.to_period('M')

weather_monthly = weather_daily.groupby('month').agg({
    'PRCP': 'mean',
    'SNOW': 'mean',
    'SNWD': 'mean',
    'TAVG': 'mean',
    'TMAX': 'mean',
    'TMIN': 'mean',
    'WT01': 'sum',
    'WT02': 'sum',
    'WT03': 'sum',
    'WT04': 'sum',
    'WT05': 'sum',
    'WT06': 'sum',
    'WT07': 'sum',
    'WT08': 'sum',
    'WT09': 'sum',
    'WT10': 'sum',
    'WT11': 'sum'
}).reset_index()
weather_monthly = weather_monthly[weather_monthly['month'] <= '2025-11']
weather_monthly = weather_monthly.rename(columns={'WT01': 'fog',
                                                  'WT02': 'heavy_fog',
                                                  'WT03': 'thunder',
                                                  'WT04': 'ice_pellets_sleet_small_hail',
                                                  'WT05': 'hail',
                                                  'WT06': 'glaze_or_rime',
                                                  'WT07': 'dust_volcanic_ash_blowing_dust_sand',
                                                  'WT08': 'smoke_or_haze',
                                                  'WT09': 'blowing_or_drifting_snow',
                                                  'WT10': 'tornado_waterspout_funnel_cloud',
                                                  'WT11': 'high_or_damaging_winds'})
weather_monthly

In [ ]:
chicago_cta_daily_boarding['service_date'] = pd.to_datetime(chicago_cta_daily_boarding['service_date'])
chicago_cta_daily_boarding = chicago_cta_daily_boarding.sort_values(by = 'service_date')
chicago_cta_daily_boarding = chicago_cta_daily_boarding[(chicago_cta_daily_boarding['service_date'].dt.year >= 2024) 
                                                        & (chicago_cta_daily_boarding['service_date'].dt.year <= 2026)]
chicago_cta_daily_boarding['month'] = chicago_cta_daily_boarding['service_date'].dt.to_period('M')
cta_monthly = chicago_cta_daily_boarding.groupby('month')[['bus', 'rail_boardings', 'total_rides']].sum().reset_index()
daytype_counts = pd.crosstab(chicago_cta_daily_boarding['service_date'].dt.to_period('M'),
                            chicago_cta_daily_boarding['day_type']).reset_index()
daytype_counts = daytype_counts.rename(columns = {'service_date' : 'month'})
chicago_cta_monthly_boarding = pd.merge(cta_monthly, daytype_counts, on = 'month')
chicago_cta_monthly_boarding = chicago_cta_monthly_boarding.rename(columns = {'A': 'total_saturdays', 
                                                                              'U' : 'sunday_or_holiday', 
                                                                              'W' : 'total_weekdays'})
chicago_cta_monthly_boarding

In [ ]:
capg = chicago_avg_price_gas.rename(columns = {'observation_date' : 'month', 'APUS23A7471A' : 'avg_gas_price'})
capg['month'] = pd.to_datetime(capg['month']).dt.to_period('M')
capg = capg[capg['month'] <= '2025-11'] # because 2025-12 and 2026-01 don't exist in chicago_cta_monthly_boarding
capg

In [14]:
# testing merging

gas = pd.read_csv(r"C:\Users\lyons\OneDrive\Desktop\is477\IS477_Tran_Machin_Lee\cleaned_data\chicago_avg_pg.csv")
cta = pd.read_csv(r"C:\Users\lyons\OneDrive\Desktop\is477\IS477_Tran_Machin_Lee\cleaned_data\chicago_cta_monthly.csv")
weather = pd.read_csv(r"C:\Users\lyons\OneDrive\Desktop\is477\IS477_Tran_Machin_Lee\cleaned_data\chicago_weather_monthly.csv")

In [15]:
merged = cta.merge(weather, on = "month", how = "left")
merged = merged.merge(gas, on = "month", how = "left")
merged

,month,bus,rail_boardings,total_rides,total_saturdays,total_sunday_or_holiday,total_weekdays,PRCP,SNOW,SNWD,...,thunder,ice_pellets_sleet_small_hail,hail,glaze_or_rime,dust_volcanic_ash_blowing_dust_sand,smoke_or_haze,blowing_or_drifting_snow,tornado_waterspout_funnel_cloud,high_or_damaging_winds,avg_gas_price
0,2024-01,12883065,8552283,21435348,4,5,22,0.128897,0.440159,2.509872,...,0.0,4.0,1.0,4.0,0.0,9.0,2.0,0.0,1.0,3.223
1,2024-02,14408475,9300019,23708494,4,4,21,0.023022,0.046524,0.065023,...,5.0,1.0,3.0,1.0,0.0,11.0,0.0,0.0,1.0,3.483
2,2024-03,14794052,10107439,24901491,5,5,21,0.139756,0.017691,0.016489,...,6.0,1.0,1.0,0.0,0.0,5.0,0.0,0.0,0.0,3.910
3,2024-04,15615211,10458339,26073550,4,4,22,0.162107,0.000590,0.000000,...,13.0,2.0,3.0,1.0,0.0,4.0,0.0,0.0,2.0,4.103
4,2024-05,16367601,11330441,27698042,4,5,22,0.147641,0.000000,0.000000,...,18.0,0.0,3.0,0.0,0.0,8.0,0.0,0.0,2.0,4.133
5,2024-06,14742769,10928671,25671440,5,5,20,0.150780,0.000000,0.000000,...,11.0,0.0,1.0,0.0,0.0,3.0,0.0,0.0,2.0,4.053
6,2024-07,15306884,11279758,26586642,4,5,22,0.175625,0.000000,0.000000,...,9.0,0.0,1.0,0.0,0.0,7.0,0.0,1.0,4.0,4.123
7,2024-08,15534600,11744693,27279293,5,4,22,0.109436,0.000000,0.000000,...,10.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,2.0,3.985
8,2024-09,16161092,11665977,27827069,4,6,20,0.069577,0.000000,0.015278,...,4.0,0.0,0.0,0.0,0.0,8.0,0.0,0.0,0.0,3.617
9,2024-10,17514190,12630875,30145065,4,4,23,0.045896,0.000000,0.000000,...,4.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,1.0,3.524


We pivoted from our original monthly Chicago gasoline-price dataset. The original dataset reported prices for all types of gasoline, but we decided this was less appropriate for our analysis for two main reasons. First, our target demographic is more closely represented by the price of regular gasoline, since that is the fuel type most likely to reflect the everyday driving costs faced by typical consumers. Second, the all-types gasoline dataset contained a missing value for October 2025, which created an avoidable data issue during merging and analysis. Because of this, we switched to a monthly Chicago regular gasoline price series, which is both more relevant to our research question and more complete for the time period we study.